In [3]:
%pip install torch_geometric statsmodels -q

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import DataLoader, TensorDataset
import torch_geometric as pyg
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data
from torch_geometric.utils import to_undirected, add_remaining_self_loops

import numpy as np
import scipy.sparse as sp
from scipy.stats import ttest_ind
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.metrics import roc_auc_score, f1_score

import matplotlib.pyplot as plt
import seaborn as sns
import json
import time
from pathlib import Path
from tqdm import tqdm
from sklearn.metrics import f1_score, precision_score, recall_score
import pandas as pd

import torch.optim as optim

# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

PROJECT_ROOT = "/kaggle/working/koopman_md17_project"
DIRS = {
    "data": os.path.join(PROJECT_ROOT, "data"),
    "checkpoints": os.path.join(PROJECT_ROOT, "checkpoints"),
    "results": os.path.join(PROJECT_ROOT, "results")
}

# Safely generate the directory tree
for dir_name, dir_path in DIRS.items():
    os.makedirs(dir_path, exist_ok=True)
    print(f"Directory established: {dir_path}")

# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

# Random seed for reproducibility
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 28.3 MB/s eta 0:00:0000:01
Note: you may need to restart the kernel to use updated packages.
Device: cpu
Directory established: /kaggle/working/koopman_md17_project/data
Directory established: /kaggle/working/koopman_md17_project/checkpoints
Directory established: /kaggle/working/koopman_md17_project/results
Device: cpu


In [4]:
"""
dataset_adapter
──────────────────
Base interface and MD17 adapter for the Koopman dynamics framework.

Usage
-----
adapter = MD17Adapter(path="/kaggle/working/.../md17_ethanol.npz",
                      molecule="ethanol",
                      sub_sampling=2,
                      window_len=150,
                      train_frac=0.8)

train_split, test_split = adapter.load()

# train_split.X      : np.ndarray  (N_train, T-1, D*2)
# train_split.y      : np.ndarray  (N_train,)   int labels
# train_split.lengths: List[int]   true lengths before padding
# train_split.meta   : dict        logging / plot titles
"""

from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Tuple, List

import numpy as np


# ─────────────────────────────────────────────────────────────────────────────
# 1. Standard data container
# ─────────────────────────────────────────────────────────────────────────────

@dataclass
class DatasetSplit:
    """
    Standard container produced by every adapter.

    Fields
    ------
    X       np.ndarray  shape (N, T, D)
                N = number of trajectory windows
                T = max sequence length across the split (shorter ones padded
                    with zeros to this length)
                D = feature dimension per timestep

    y       np.ndarray  shape (N,)  int64
                Class label per trajectory.
                If the dataset has no meaningful labels, this is all zeros.

    lengths List[int]   length N
                True (unpadded) length of each trajectory.
                The model uses these to ignore the zero-padded tail.

    meta    dict
                Dataset-specific metadata for logging and plot titles.
                Guaranteed keys: 'dataset', 'molecule', 'D', 'split'
    """
    X      : np.ndarray
    y      : np.ndarray
    lengths: List[int]
    meta   : dict = field(default_factory=dict)


# ─────────────────────────────────────────────────────────────────────────────
# 2. Abstract base class — every new dataset subclasses this
# ─────────────────────────────────────────────────────────────────────────────

class DynamicsDatasetAdapter(ABC):
    """
    Contract all dataset adapters must satisfy.

    The training loop and evaluator only call:
        train_split, test_split = adapter.load()
        model = SomeModel(input_dim=adapter.input_dim, ...)

    They never touch raw files, windowing logic, or label encodings.
    """

    @abstractmethod
    def load(self) -> Tuple[DatasetSplit, DatasetSplit]:
        """
        Parse raw data, compute features, window, pad, split.
        Returns (train_split, test_split).
        """
        pass

    @property
    @abstractmethod
    def input_dim(self) -> int:
        """
        Feature dimension D seen by the model.

        Must be computable BEFORE load() is called so the model can be
        instantiated before loading data into memory.
        """
        pass

    @property
    @abstractmethod
    def name(self) -> str:
        """Human-readable dataset name used in log lines and plot titles."""
        pass

In [5]:
"""
dataset_adapter.py
──────────────────
Base interface and MD17 adapter for the Koopman dynamics framework.

Usage
-----
adapter = MD17Adapter(path="/kaggle/working/.../md17_ethanol.npz",
                      molecule="ethanol",
                      sub_sampling=2,
                      window_len=150,
                      train_frac=0.8)

train_split, test_split = adapter.load()

# train_split.X      : np.ndarray  (N_train, T-1, D*2)
# train_split.y      : np.ndarray  (N_train,)   int labels
# train_split.lengths: List[int]   true lengths before padding
# train_split.meta   : dict        logging / plot titles
"""

from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Tuple, List

import numpy as np


# ─────────────────────────────────────────────────────────────────────────────
# 1. Standard data container
# ─────────────────────────────────────────────────────────────────────────────

@dataclass
class DatasetSplit:
    """
    Standard container produced by every adapter.

    Fields
    ------
    X       np.ndarray  shape (N, T, D)
                N = number of trajectory windows
                T = max sequence length across the split (shorter ones padded
                    with zeros to this length)
                D = feature dimension per timestep

    y       np.ndarray  shape (N,)  int64
                Class label per trajectory.
                If the dataset has no meaningful labels, this is all zeros.

    lengths List[int]   length N
                True (unpadded) length of each trajectory.
                The model uses these to ignore the zero-padded tail.

    meta    dict
                Dataset-specific metadata for logging and plot titles.
                Guaranteed keys: 'dataset', 'molecule', 'D', 'split'
    """
    X      : np.ndarray
    y      : np.ndarray
    lengths: List[int]
    meta   : dict = field(default_factory=dict)


# ─────────────────────────────────────────────────────────────────────────────
# 2. Abstract base class — every new dataset subclasses this
# ─────────────────────────────────────────────────────────────────────────────

class DynamicsDatasetAdapter(ABC):
    """
    Contract all dataset adapters must satisfy.

    The training loop and evaluator only call:
        train_split, test_split = adapter.load()
        model = SomeModel(input_dim=adapter.input_dim, ...)

    They never touch raw files, windowing logic, or label encodings.
    """

    @abstractmethod
    def load(self) -> Tuple[DatasetSplit, DatasetSplit]:
        """
        Parse raw data, compute features, window, pad, split.
        Returns (train_split, test_split).
        """
        pass

    @property
    @abstractmethod
    def input_dim(self) -> int:
        """
        Feature dimension D seen by the model.

        Must be computable BEFORE load() is called so the model can be
        instantiated before loading data into memory.
        """
        pass

    @property
    @abstractmethod
    def name(self) -> str:
        """Human-readable dataset name used in log lines and plot titles."""
        pass


# ─────────────────────────────────────────────────────────────────────────────
# 3. MD17 adapter
# ─────────────────────────────────────────────────────────────────────────────

class MD17Adapter(DynamicsDatasetAdapter):
    """
    Adapter for the MD17 molecular dynamics benchmark.

    Raw file format
    ---------------
    .npz with keys: 'R' (coordinates), 'E' (energies), optionally 'F' (forces)
        R : (n_frames, n_atoms, 3)   atomic positions in Angstrom
        E : (n_frames,)              potential energy in kcal/mol

    Feature construction (mirrors the original notebook pipeline)
    ─────────────────────────────────────────────────────────────
    Step 1 — Sub-sample:  take every `sub_sampling`-th frame from R and E.

    Step 2 — Pairwise distances:  for each frame, compute the upper triangle
             of the pairwise Euclidean distance matrix.
             flat_dim = n_atoms * (n_atoms - 1) // 2
             This gives rotational and translational invariance.

    Step 3 — Window:  cut the sub-sampled trajectory into non-overlapping
             segments of length `window_len`.  Each segment becomes one
             trajectory in the dataset.

    Step 4 — Label:  each window is labeled 0 (low energy) or 1 (high energy)
             based on whether its mean energy is above the median.
             This matches the original notebook convention.

    Step 5 — Phase-space velocity:  inside build_phase_space() below,
             velocity at time t is defined as the finite difference
                 v_t = x_{t+1} - x_t
             and the feature at time t is [x_{t+1} || v_t].
             This consumes one frame, so the output length is window_len - 1.

    Step 6 — Pad:  within each split, pad all trajectories to the same length
             with zeros so they can be stacked into a single ndarray.

    Parameters
    ----------
    path         : str   path to the .npz file
    molecule     : str   molecule name, used only in metadata
    sub_sampling : int   take every N-th frame  (default 2)
    window_len   : int   frames per trajectory window  (default 150)
    train_frac   : float fraction of windows used for training  (default 0.8)
    """

    # atom counts for common MD17 molecules — used by input_dim
    _ATOM_COUNTS = {
        "aspirin"      : 21,
        "benzene"      : 12,
        "ethanol"      :  9,
        "malonaldehyde": 9,
        "naphthalene"  : 18,
        "salicylic"    : 16,
        "toluene"      : 15,
        "uracil"       : 12,
    }

    def __init__(
        self,
        path        : str,
        molecule    : str  = "ethanol",
        sub_sampling: int  = 2,
        window_len  : int  = 150,
        train_frac  : float = 0.8,
    ):
        self.path         = path
        self.molecule     = molecule.lower()
        self.sub_sampling = sub_sampling
        self.window_len   = window_len
        self.train_frac   = train_frac

        # Resolve n_atoms — try the lookup table first, fall back to the file
        if self.molecule in self._ATOM_COUNTS:
            self._n_atoms = self._ATOM_COUNTS[self.molecule]
        else:
            # Will be set properly during load()
            self._n_atoms = None

    # ── Public properties ────────────────────────────────────────────────────

    @property
    def name(self) -> str:
        return f"MD17-{self.molecule}"

    @property
    def input_dim(self) -> int:
        """
        D = 2 * flat_dim  because each timestep = [position || velocity]
        and both have the same shape (upper triangle of distance matrix).

        flat_dim = n_atoms * (n_atoms - 1) // 2
        """
        if self._n_atoms is None:
            raise RuntimeError(
                "n_atoms unknown for molecule '{}'. "
                "Call load() first or add it to _ATOM_COUNTS.".format(self.molecule)
            )
        flat_dim = self._n_atoms * (self._n_atoms - 1) // 2
        return 2 * flat_dim

    # ── Main entry point ─────────────────────────────────────────────────────

    def load(self) -> Tuple[DatasetSplit, DatasetSplit]:
        """
        Full pipeline: raw file → (train_split, test_split).
        """

        # ── Step 1: read raw file ────────────────────────────────────────────
        # Supports both rMD17 ('coords'/'energies') and
        # older MD17 ('R'/'E') key conventions.
        raw = np.load(self.path, allow_pickle=True)

        coords   = raw["coords"]   if "coords"   in raw else raw["R"]
        energies = raw["energies"] if "energies" in raw else raw["E"]

        # rMD17 stores n_atoms directly in nuclear_charges
        if "nuclear_charges" in raw:
            self._n_atoms = int(raw["nuclear_charges"].shape[0])

        n_frames, n_atoms, _ = coords.shape
        if self._n_atoms is None:
            self._n_atoms = n_atoms    # fallback if not set from nuclear_charges

        # ── Step 2: sub-sample ───────────────────────────────────────────────
        coords   = coords[::self.sub_sampling]
        energies = energies[::self.sub_sampling]

        # ── Step 3: pairwise distances ───────────────────────────────────────
        #
        # For each frame, compute the upper triangle of the n_atoms x n_atoms
        # Euclidean distance matrix.
        #
        # diff[i,j] = coord_i - coord_j  →  shape (n_frames, n_atoms, n_atoms, 3)
        # We use vectorized broadcasting across all frames at once to avoid
        # a Python loop over frames, which would be slow for large datasets.
        #
        diff = coords[:, :, None, :] - coords[:, None, :, :]
        # diff shape: (n_frames, n_atoms, n_atoms, 3)

        dist_matrix = np.sqrt(np.sum(diff ** 2, axis=-1))
        # dist_matrix shape: (n_frames, n_atoms, n_atoms)

        triu_idx  = np.triu_indices(n_atoms, k=1)
        flat_dist = dist_matrix[:, triu_idx[0], triu_idx[1]]
        # flat_dist shape: (n_frames, flat_dim)
        # flat_dim = n_atoms * (n_atoms - 1) // 2

        # ── Step 4: window + label ───────────────────────────────────────────
        trajectories, labels = self._make_windows(flat_dist, energies)

        # ── Step 5: train / test split ───────────────────────────────────────
        split_idx = int(len(trajectories) * self.train_frac)
        train_trajs  = trajectories[:split_idx]
        train_labels = labels[:split_idx]
        test_trajs   = trajectories[split_idx:]
        test_labels  = labels[split_idx:]

        # ── Step 6: phase-space velocity + pad ──────────────────────────────
        train_split = self._build_split(train_trajs, train_labels, split="train")
        test_split  = self._build_split(test_trajs,  test_labels,  split="test")

        return train_split, test_split

    # ── Private helpers ──────────────────────────────────────────────────────

    def _make_windows(
        self,
        flat_dist: np.ndarray,   # (n_frames, flat_dim)
        energies : np.ndarray,   # (n_frames,)
    ) -> Tuple[List[np.ndarray], List[int]]:
        """
        Cut flat_dist into non-overlapping windows of length window_len.
        Label each window 1 (high energy) or 0 (low energy) relative to
        the global median — matching the original notebook convention.

        Returns
        -------
        trajectories : List of np.ndarray, each shape (window_len, flat_dim)
        labels       : List of int  (0 or 1)
        """
        total_steps = flat_dist.shape[0]
        # Integer division — the tail that does not fill a complete window
        # is discarded, matching the original notebook.
        n_windows   = (total_steps - self.window_len) // self.window_len

        threshold    = np.median(energies)
        trajectories = []
        labels       = []

        for idx in range(n_windows):
            start = idx * self.window_len
            end   = start + self.window_len

            trajectories.append(flat_dist[start:end])   # (window_len, flat_dim)
            mean_energy = np.mean(energies[start:end])
            labels.append(1 if mean_energy > threshold else 0)

        return trajectories, labels

    def _build_split(
        self,
        trajectories: List[np.ndarray],   # each (window_len, flat_dim)
        labels      : List[int],
        split       : str,
    ) -> DatasetSplit:
        """
        Apply phase-space velocity construction and zero-pad to a fixed
        length within this split.

        Phase-space step
        ─────────────────
        For trajectory of shape (T, flat_dim):
            position at step t  = traj[t+1]          shape (flat_dim,)
            velocity at step t  = traj[t+1] - traj[t] shape (flat_dim,)
            feature at step t   = [position || velocity]  shape (2*flat_dim,)

        This produces a sequence of length T-1 = window_len - 1.
        The -1 is why lengths stores window_len - 1, not window_len.

        Padding
        ────────
        Within the split, all sequences are padded with zeros to max_len
        (= window_len - 1, which is the same for every window since all
        windows have the same length).  The padding logic is written to
        handle variable-length sequences too — this will matter when we
        add datasets with variable trajectory lengths.
        """
        phase_sequences = []
        true_lengths    = []

        for traj in trajectories:
            # traj: (window_len, flat_dim)
            pos = traj[1:]          # (window_len-1, flat_dim)  — aligned positions
            vel = traj[1:] - traj[:-1]  # (window_len-1, flat_dim)  — finite diff velocity
            phase = np.concatenate([pos, vel], axis=-1)  # (window_len-1, 2*flat_dim)
            phase_sequences.append(phase)
            true_lengths.append(phase.shape[0])

        # Pad to max length within this split
        max_len  = max(true_lengths)
        flat_dim = trajectories[0].shape[1]
        D        = 2 * flat_dim

        X_padded = np.zeros((len(phase_sequences), max_len, D), dtype=np.float32)
        for i, seq in enumerate(phase_sequences):
            X_padded[i, :seq.shape[0], :] = seq

        y = np.array(labels, dtype=np.int64)

        meta = {
            "dataset"      : "MD17",
            "molecule"     : self.molecule,
            "D"            : D,
            "flat_dim"     : flat_dim,
            "n_atoms"      : self._n_atoms,
            "window_len"   : self.window_len,
            "sub_sampling" : self.sub_sampling,
            "split"        : split,
            "n_trajectories": len(phase_sequences),
        }

        return DatasetSplit(
            X       = X_padded,
            y       = y,
            lengths = true_lengths,
            meta    = meta,
        )

In [6]:
# Drop this at the top of your notebook (or import from file)
adapter = MD17Adapter(
    path         = "/kaggle/input/datasets/abhilash437/md17-ethanol/rmd17_ethanol.npz",
    molecule     = "ethanol",
    sub_sampling = 2,
    window_len   = 150,
    train_frac   = 0.8,
)

train_split, test_split = adapter.load()

# These replace your current X_mol_train, y_mol_train, etc.
X_mol_train  = train_split.X
y_mol_train  = train_split.y
# lengths are inside train_split.lengths — pass to build_velocity_batch as before

print(adapter.input_dim)   # should print 72 for ethanol
print(train_split.meta)

72
{'dataset': 'MD17', 'molecule': 'ethanol', 'D': 72, 'flat_dim': 36, 'n_atoms': 9, 'window_len': 150, 'sub_sampling': 2, 'split': 'train', 'n_trajectories': 265}


In [7]:
# Quick sanity check after loading
train_split, test_split = adapter.load()

print(f"input_dim      : {adapter.input_dim}")          # expect 72
print(f"train X shape  : {train_split.X.shape}")        # (N_train, 149, 72)
print(f"test  X shape  : {test_split.X.shape}")         # (N_test,  149, 72)
print(f"train lengths  : {set(train_split.lengths)}")   # should be {149}
print(f"meta           : {train_split.meta}")

input_dim      : 72
train X shape  : (265, 149, 72)
test  X shape  : (67, 149, 72)
train lengths  : {149}
meta           : {'dataset': 'MD17', 'molecule': 'ethanol', 'D': 72, 'flat_dim': 36, 'n_atoms': 9, 'window_len': 150, 'sub_sampling': 2, 'split': 'train', 'n_trajectories': 265}


In [8]:
"""
evaluator.py
────────────
Dataset-agnostic evaluator for the Koopman dynamics framework.

Usage
-----
evaluator = KoopmanEvaluator(
    koopman_model  = anchored_block_model,
    baseline_model = gru_model,
    device         = 'cpu',
    rollout_steps  = 29,
    sub_dim        = 16,        # size of the kinetic subspace
)

results = evaluator.run(test_split)

# results is an EvalResults dataclass — contains all four metric groups.
# Pass it to evaluator.print_summary(results) for a readable report.
# Pass it to evaluator.plot(results, title="MD17 Ethanol") for the tradeoff plot.

Key design decision
-------------------
The adapter's DatasetSplit.X is already in phase-space form
(position || velocity, shape (N, T, D)).  The evaluator feeds it
directly to the model — it does NOT call build_velocity_batch.
That function is retired in this framework.
"""

from dataclasses import dataclass, field
from typing import Optional, List, Dict, Any

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# from dataset_adapter import DatasetSplit


# ─────────────────────────────────────────────────────────────────────────────
# 1. Results container
# ─────────────────────────────────────────────────────────────────────────────

@dataclass
class EvalResults:
    """
    All metrics produced by one evaluator.run() call.

    Metric groups
    ─────────────
    rollout   : MSE at each prediction horizon step (mean ± std over sequences)
    spectral  : spectral radii of the Koopman operator blocks
    geometry  : pairwise latent distance retention over the rollout horizon
    collapse  : relative change metric confirming the encoder is not collapsed
    meta      : passed through from DatasetSplit.meta for logging / plot titles
    """
    # Rollout MSE — shape (rollout_steps,) for mean, same for std
    koopman_mse_mean : np.ndarray
    koopman_mse_std  : np.ndarray
    baseline_mse_mean: np.ndarray
    baseline_mse_std : np.ndarray

    # Spectral radii
    rho_kin  : float
    rho_pheno: float

    # Geometry retention — normalized pairwise distance ratio
    # shape (rollout_steps + 1,) — index 0 is t=0 (always 1.0 after normalization)
    koopman_geom_mean : np.ndarray
    koopman_geom_std  : np.ndarray
    baseline_geom_mean: np.ndarray
    baseline_geom_std : np.ndarray

    # Collapse diagnostic
    relative_change_koopman : float
    relative_change_baseline: float

    # Metadata from the DatasetSplit
    meta: dict = field(default_factory=dict)


# ─────────────────────────────────────────────────────────────────────────────
# 2. Evaluator
# ─────────────────────────────────────────────────────────────────────────────

class KoopmanEvaluator:
    """
    Computes all four metric groups for a Koopman model vs a baseline model
    on any DatasetSplit produced by a DynamicsDatasetAdapter.

    Parameters
    ----------
    koopman_model   : nn.Module
        Must expose:
            - encoder_kin(x_flat)   → z_kin   (N, sub_dim)
            - encoder_pheno(x_flat) → z_pheno (N, sub_dim)
            - K                     → (latent_dim, latent_dim) tensor
              where K[:sub_dim, :sub_dim] is the kinetic block

    baseline_model  : nn.Module
        Must expose:
            - encoder(x_flat) or encoder_shared(x_flat) → z (N, latent_dim)
            - K                                         → (latent_dim, latent_dim)
              (or the model implements forward_rollout for GRU-style baseline)

    device          : str   'cpu' or 'cuda'
    rollout_steps   : int   number of autoregressive steps to evaluate
    sub_dim         : int   size of kinetic subspace (default 16)
    batch_size      : int   sequences processed at a time during encoding
    """

    def __init__(
        self,
        koopman_model  : nn.Module,
        baseline_model : nn.Module,
        device         : str = 'cpu',
        rollout_steps  : int = 29,
        sub_dim        : int = 16,
        batch_size     : int = 64,
    ):
        self.koopman_model  = koopman_model.to(device).eval()
        self.baseline_model = baseline_model.to(device).eval()
        self.device         = device
        self.rollout_steps  = rollout_steps
        self.sub_dim        = sub_dim
        self.batch_size     = batch_size

    # ── Public entry point ───────────────────────────────────────────────────

    def run(self, split: DatasetSplit) -> EvalResults:
        """
        Full evaluation pipeline on one DatasetSplit.
        Returns an EvalResults dataclass.
        """
        X       = split.X        # (N, T, D)  numpy, already phase-space form
        lengths = split.lengths  # List[int]

        # Convert to torch — the evaluator owns the device placement
        X_t = torch.tensor(X, dtype=torch.float32, device=self.device)

        with torch.no_grad():
            z_koop, z_base = self._encode_all(X_t, lengths)
            # z_koop: (N, T, latent_dim)  — cat(z_kin, z_pheno)
            # z_base: (N, T, latent_dim)

            K_koop = self.koopman_model.K.detach().cpu()
            K_base = self._get_baseline_K()

            koop_mse_mean, koop_mse_std   = self._rollout_mse(z_koop, lengths, K_koop)
            base_mse_mean, base_mse_std   = self._rollout_mse(z_base, lengths, K_base)

            rho_kin, rho_pheno            = self._spectral_radii(K_koop)

            koop_geom_mean, koop_geom_std = self._geometry_retention(z_koop, lengths, K_koop)
            base_geom_mean, base_geom_std = self._geometry_retention(z_base, lengths, K_base)

            rc_koop = self._relative_change(z_koop, lengths)
            rc_base = self._relative_change(z_base, lengths)

        return EvalResults(
            koopman_mse_mean        = koop_mse_mean,
            koopman_mse_std         = koop_mse_std,
            baseline_mse_mean       = base_mse_mean,
            baseline_mse_std        = base_mse_std,
            rho_kin                 = rho_kin,
            rho_pheno               = rho_pheno,
            koopman_geom_mean       = koop_geom_mean,
            koopman_geom_std        = koop_geom_std,
            baseline_geom_mean      = base_geom_mean,
            baseline_geom_std       = base_geom_std,
            relative_change_koopman = rc_koop,
            relative_change_baseline= rc_base,
            meta                    = split.meta,
        )

    # ── Encoding ─────────────────────────────────────────────────────────────

    def _encode_all(self, X_t, lengths):
        """
        Encode every frame of every sequence through both models.

        X_t : (N, T, D) torch tensor on self.device
        Returns z_koop (N, T, latent_dim), z_base (N, T, latent_dim)

        Processing in batches of self.batch_size to avoid OOM on large datasets.
        Frames are encoded flat (N*T, D) then reshaped back to (N, T, latent_dim).
        Padding frames are encoded too but ignored downstream via the lengths mask.
        """
        N, T, D = X_t.shape
        x_flat  = X_t.view(N * T, D)   # (N*T, D)

        # Encode in chunks
        z_kin_chunks   = []
        z_pheno_chunks = []
        z_base_chunks  = []

        for start in range(0, N * T, self.batch_size * T):
            end   = min(start + self.batch_size * T, N * T)
            chunk = x_flat[start:end]

            z_kin_chunks.append(self.koopman_model.encoder_kin(chunk))
            z_pheno_chunks.append(self.koopman_model.encoder_pheno(chunk))
            z_base_chunks.append(self._encode_baseline(chunk))

        z_kin   = torch.cat(z_kin_chunks,   dim=0).view(N, T, -1)
        z_pheno = torch.cat(z_pheno_chunks, dim=0).view(N, T, -1)
        z_koop  = torch.cat([z_kin, z_pheno], dim=-1)   # (N, T, latent_dim)
        z_base  = torch.cat(z_base_chunks,  dim=0).view(N, T, -1)

        return z_koop, z_base

    def _encode_baseline(self, x_flat):
        """
        Handles both naming conventions for the baseline encoder:
            BlackBoxGraphDynamicsNet  uses  .encoder
            TrajectoryWindowedKoopmanNet uses .encoder_shared
        """
        if hasattr(self.baseline_model, 'encoder'):
            return self.baseline_model.encoder(x_flat)
        elif hasattr(self.baseline_model, 'encoder_shared'):
            return self.baseline_model.encoder_shared(x_flat)
        else:
            raise AttributeError(
                "baseline_model must have an 'encoder' or 'encoder_shared' attribute."
            )

    def _get_baseline_K(self):
        """
        Retrieves the baseline transition matrix K.
        For GRU-style models that don't have a K attribute, we compute
        an empirical linear approximation via least squares on the encoded
        latent trajectories — this gives a fair spectral comparison.
        """
        if hasattr(self.baseline_model, 'K'):
            return self.baseline_model.K.detach().cpu()
        else:
            # GRU has no K — return None; rollout will use forward_rollout instead
            return None

    # ── Metric: rollout MSE ──────────────────────────────────────────────────

    def _rollout_mse(self, z, lengths, K):
        """
        Autoregressive multi-step rollout MSE.

        z       : (N, T, latent_dim) encoded trajectories
        lengths : List[int]          true lengths per sequence
        K       : (latent_dim, latent_dim) or None (GRU path)

        For each horizon step s in 1..rollout_steps:
            pred_t = z_t @ K^s^T
            error  = MSE(pred_t, z_{t+s})  averaged over all valid (b, t) pairs

        Key term — matrix power K^s:
            Applying K once predicts one step ahead. Applying it s times
            (matrix_power) predicts s steps ahead in one operation, which is
            equivalent to s sequential applications but cheaper to compute.

        Returns (mean_errors, std_errors) each shape (rollout_steps,)
        """
        z_cpu = z.cpu()
        mse_per_step_mean = np.zeros(self.rollout_steps)
        mse_per_step_std  = np.zeros(self.rollout_steps)

        if K is None:
            # GRU path — use forward_rollout if available
            return self._rollout_mse_gru(z, lengths)

        for s in range(1, self.rollout_steps + 1):
            K_pow = torch.matrix_power(K, s)   # (latent_dim, latent_dim)
            errors = []

            for b, T in enumerate(lengths):
                if T <= s + 1:
                    continue
                # Predict z_{t+s} from z_t using K^s
                z_init   = z_cpu[b, :T - s]           # (T-s, latent_dim)
                z_target = z_cpu[b, s:T]               # (T-s, latent_dim)
                z_pred   = torch.matmul(z_init, K_pow.t())

                mse = torch.nn.functional.mse_loss(z_pred, z_target).item()
                errors.append(mse)

            mse_per_step_mean[s - 1] = np.mean(errors) if errors else 0.0
            mse_per_step_std[s - 1]  = np.std(errors)  if errors else 0.0

        return mse_per_step_mean, mse_per_step_std

    def _rollout_mse_gru(self, z, lengths):
        """
        Fallback rollout for GRU baseline using forward_rollout.
        Uses the encoded starting state z_0 as seed and rolls out autoregressively.
        """
        z_cpu = z.cpu()
        mse_per_step_mean = np.zeros(self.rollout_steps)
        mse_per_step_std  = np.zeros(self.rollout_steps)

        for s in range(1, self.rollout_steps + 1):
            errors = []
            for b, T in enumerate(lengths):
                if T <= s + 1:
                    continue
                # Use the GRU model's own rollout method for fairness
                seed = z_cpu[b:b+1, :1, :]   # (1, 1, latent_dim) — first frame
                preds = self.baseline_model.forward_rollout(
                    seed.to(self.device), steps=s + 1
                ).cpu()
                z_pred   = preds[0, s]                   # (latent_dim,)
                z_target = z_cpu[b, s]                   # (latent_dim,)
                errors.append(torch.nn.functional.mse_loss(z_pred, z_target).item())

            mse_per_step_mean[s - 1] = np.mean(errors) if errors else 0.0
            mse_per_step_std[s - 1]  = np.std(errors)  if errors else 0.0

        return mse_per_step_mean, mse_per_step_std

    # ── Metric: spectral radii ───────────────────────────────────────────────

    def _spectral_radii(self, K):
        """
        Computes spectral radius of the kinetic block (expected 1.0)
        and the phenotypic block.

        Key term — spectral radius ρ(A):
            The largest absolute eigenvalue of matrix A.
            ρ = 1.0  →  conservative (volume-preserving) dynamics
            ρ < 1.0  →  dissipative (contracting) dynamics
            ρ > 1.0  →  expansive (diverging) dynamics
        """
        K_np       = K.numpy()
        K_kin      = K_np[:self.sub_dim, :self.sub_dim]
        K_pheno    = K_np[self.sub_dim:, self.sub_dim:]
        eigvals    = np.linalg.eigvals

        rho_kin   = float(np.max(np.abs(eigvals(K_kin))))
        rho_pheno = float(np.max(np.abs(eigvals(K_pheno))))

        return rho_kin, rho_pheno

    # ── Metric: geometry retention ───────────────────────────────────────────

    def _geometry_retention(self, z, lengths, K):
        """
        Pairwise latent distance retention over the rollout horizon.

        At each step s, for a randomly sampled pair of sequences (i, j),
        compute the ratio of predicted pairwise distance to true pairwise
        distance at t=0:

            ratio_s = ||pred_i(s) - pred_j(s)|| / ||z_i(0) - z_j(0)||

        A ratio of 1.0 means the operator perfectly preserved pairwise geometry.
        Ratio → 0 means the operator collapsed both trajectories to the same point
        (the GRU attractor trap).

        Key term — isometry:
            A map f is an isometry if it preserves all pairwise distances.
            An orthogonal operator K is an isometry by construction:
            ||Kx - Ky|| = ||K(x-y)|| = ||x-y||  (because K preserves norms).

        Returns (geom_mean, geom_std) each shape (rollout_steps + 1,)
        Index 0 = t=0 (always 1.0 before normalization).
        """
        z_cpu  = z.cpu()
        N      = len(lengths)
        n_pairs = min(200, N * (N - 1) // 2)   # cap for speed

        # Sample random pairs
        rng   = np.random.default_rng(42)
        idx_i = rng.integers(0, N, size=n_pairs)
        idx_j = rng.integers(0, N, size=n_pairs)
        # Avoid self-pairs
        mask  = idx_i != idx_j
        idx_i, idx_j = idx_i[mask], idx_j[mask]

        geom_means = np.zeros(self.rollout_steps + 1)
        geom_stds  = np.zeros(self.rollout_steps + 1)

        if K is None:
            # GRU has no K — return zeros (handled in plot as N/A)
            return geom_means, geom_stds

        for s in range(self.rollout_steps + 1):
            if s == 0:
                K_pow = torch.eye(z_cpu.shape[-1])
            else:
                K_pow = torch.matrix_power(K, s)

            ratios = []
            for i, j in zip(idx_i, idx_j):
                T_i = min(lengths[i], lengths[j])
                if T_i < 1:
                    continue

                # Use first timestep as reference
                z_i0 = z_cpu[i, 0]   # (latent_dim,)
                z_j0 = z_cpu[j, 0]   # (latent_dim,)
                d0   = torch.norm(z_i0 - z_j0).item()
                if d0 < 1e-8:
                    continue

                # Predict both trajectories s steps forward
                z_i_pred = torch.matmul(z_i0.unsqueeze(0), K_pow.t()).squeeze()
                z_j_pred = torch.matmul(z_j0.unsqueeze(0), K_pow.t()).squeeze()
                d_pred   = torch.norm(z_i_pred - z_j_pred).item()

                ratios.append(d_pred / d0)

            geom_means[s] = np.mean(ratios) if ratios else 0.0
            geom_stds[s]  = np.std(ratios)  if ratios else 0.0

        return geom_means, geom_stds

    # ── Metric: collapse diagnostic ──────────────────────────────────────────

    def _relative_change(self, z, lengths):
        """
        Scale-invariant collapse indicator.

        relative_change = mean over all (b,t): ||z_{t+1} - z_t|| / ||z_t||

        > 0.05 → encoder is producing dynamic (non-collapsed) trajectories
        ≈ 0    → fixed-point collapse (all inputs map to the same vector)

        Key term — fixed-point collapse:
            The encoder maps all inputs to nearly the same latent vector.
            This trivially minimises the dynamics loss (K≈I satisfies it)
            but destroys all representational content.
        """
        z_cpu = z.cpu()
        all_ratios = []

        for b, T in enumerate(lengths):
            if T < 2:
                continue
            z_seq    = z_cpu[b, :T]                             # (T, latent_dim)
            diffs    = (z_seq[1:] - z_seq[:-1]).norm(dim=1)     # (T-1,)
            norms    = z_seq[:-1].norm(dim=1)                   # (T-1,)
            ratios   = (diffs / (norms + 1e-6)).tolist()
            all_ratios.extend(ratios)

        return float(np.mean(all_ratios)) if all_ratios else 0.0

    # ── Reporting ────────────────────────────────────────────────────────────

    def print_summary(self, results: EvalResults):
        """Prints a readable summary of all four metric groups."""
        sep = "=" * 70
        print(sep)
        print(f"EVALUATION SUMMARY — {results.meta.get('dataset','?')} "
              f"{results.meta.get('molecule','')}")
        print(sep)

        print("\n[1] ROLLOUT MSE")
        print(f"  {'Step':>4}  {'Koopman':>10}  {'Baseline':>10}")
        for s in range(len(results.koopman_mse_mean)):
            print(f"  {s+1:>4}  "
                  f"{results.koopman_mse_mean[s]:>10.4f}  "
                  f"{results.baseline_mse_mean[s]:>10.4f}")

        print("\n[2] SPECTRAL RADII")
        print(f"  rho(K_kin)  : {results.rho_kin:.6f}  "
              f"{'PASS (conservative)' if abs(results.rho_kin - 1.0) < 1e-3 else 'WARN'}")
        print(f"  rho(K_pheno): {results.rho_pheno:.4f}")

        final_koop = results.koopman_geom_mean[-1]
        final_base = results.baseline_geom_mean[-1]
        print("\n[3] GEOMETRY RETENTION @ final step")
        print(f"  Koopman : {final_koop:.4f}")
        print(f"  Baseline: {final_base:.4f}")

        print("\n[4] COLLAPSE DIAGNOSTIC")
        print(f"  Koopman  relative change: {results.relative_change_koopman:.4f}  "
              f"{'PASS' if results.relative_change_koopman > 0.05 else 'FAIL'}")
        print(f"  Baseline relative change: {results.relative_change_baseline:.4f}")
        print(sep)

    def plot(self, results: EvalResults, title: Optional[str] = None,
             save_path: Optional[str] = None):
        """
        Two-panel tradeoff plot:
          Top    — Rollout MSE vs prediction horizon
          Bottom — Normalized pairwise geometry retention vs horizon
        """
        KOOP_COLOR = "#2166ac"
        BASE_COLOR = "#d6604d"

        steps_err  = np.arange(1, self.rollout_steps + 1)
        steps_geom = np.arange(0, self.rollout_steps + 1)

        # Normalize geometry to t=0
        k0 = results.koopman_geom_mean[0] + 1e-8
        b0 = results.baseline_geom_mean[0] + 1e-8
        koop_geom_norm = results.koopman_geom_mean  / k0
        base_geom_norm = results.baseline_geom_mean / b0
        koop_gstd_norm = results.koopman_geom_std   / k0
        base_gstd_norm = results.baseline_geom_std  / b0

        dataset_label = (title or
                         f"{results.meta.get('dataset','?')} "
                         f"{results.meta.get('molecule','')}")

        fig = plt.figure(figsize=(13, 8))
        gs  = gridspec.GridSpec(2, 1, hspace=0.08)

        # ── Top: MSE ──────────────────────────────────────────────────────
        ax1 = fig.add_subplot(gs[0])
        ax1.plot(steps_err, results.koopman_mse_mean,
                 color=KOOP_COLOR, linewidth=2.5, marker='o', markersize=4,
                 label='Koopman (Orthogonal)')
        ax1.fill_between(steps_err,
                         results.koopman_mse_mean - results.koopman_mse_std,
                         results.koopman_mse_mean + results.koopman_mse_std,
                         color=KOOP_COLOR, alpha=0.15)
        ax1.plot(steps_err, results.baseline_mse_mean,
                 color=BASE_COLOR, linewidth=2.5, linestyle='--',
                 marker='x', markersize=5, label='GRU Baseline')
        ax1.fill_between(steps_err,
                         results.baseline_mse_mean - results.baseline_mse_std,
                         results.baseline_mse_mean + results.baseline_mse_std,
                         color=BASE_COLOR, alpha=0.15)
        ax1.set_ylabel("Rollout MSE", fontsize=12)
        ax1.set_title(
            f"Predictive Accuracy vs. Geometric Stability — {dataset_label}",
            fontsize=13, fontweight='bold', pad=10)
        ax1.legend(fontsize=11, loc='upper left')
        ax1.grid(True, linestyle=':', alpha=0.5)
        ax1.set_xticklabels([])

        # ── Bottom: geometry retention ────────────────────────────────────
        ax2 = fig.add_subplot(gs[1], sharex=ax1)
        ax2.plot(steps_geom, koop_geom_norm,
                 color=KOOP_COLOR, linewidth=2.5, marker='o', markersize=4)
        ax2.fill_between(steps_geom,
                         koop_geom_norm - koop_gstd_norm,
                         koop_geom_norm + koop_gstd_norm,
                         color=KOOP_COLOR, alpha=0.15)
        ax2.plot(steps_geom, base_geom_norm,
                 color=BASE_COLOR, linewidth=2.5, linestyle='--', marker='x',
                 markersize=5)
        ax2.fill_between(steps_geom,
                         base_geom_norm - base_gstd_norm,
                         base_geom_norm + base_gstd_norm,
                         color=BASE_COLOR, alpha=0.15)
        ax2.axhline(1.0, color='gray', linewidth=1.0, linestyle=':',
                    alpha=0.7, label='Perfect retention (ratio = 1.0)')
        ax2.set_xlabel("Prediction Horizon (steps)", fontsize=12)
        ax2.set_ylabel("Pairwise Distance Ratio\n(normalized to t=0)", fontsize=12)
        ax2.legend(fontsize=10, loc='lower left')
        ax2.grid(True, linestyle=':', alpha=0.5)

        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.show()

        # Print key numbers
        print("\nTRADEOFF SUMMARY")
        print(f"{'Metric':<40} {'Koopman':>10} {'Baseline':>10}")
        print("-" * 62)
        print(f"{'MSE @ step 1 (short horizon)':<40} "
              f"{results.koopman_mse_mean[0]:>10.4f} "
              f"{results.baseline_mse_mean[0]:>10.4f}")
        print(f"{'MSE @ final step':<40} "
              f"{results.koopman_mse_mean[-1]:>10.4f} "
              f"{results.baseline_mse_mean[-1]:>10.4f}")
        print(f"{'Geometry retention @ final step':<40} "
              f"{koop_geom_norm[-1]:>10.4f} "
              f"{base_geom_norm[-1]:>10.4f}")